<a href="https://colab.research.google.com/github/Zayaiken21/Youtube-Blue/blob/main/chatterbox_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================================
# YOUTUBE BLUE — CELL 1
# CHATTERBOX TURBO INSTALL / REPAIR
# ======================================================================
#
# RUN THIS:
# - on a brand-new Colab runtime
# - if Cell 2 says the environment is missing/broken
#
# DO NOT run this every time just to restart the backend.
# Cell 2 is the normal fast startup.
#
# Fixes included:
# - Python 3.10 isolated environment
# - Chatterbox-TTS-Server
# - chatterbox-v2 installed correctly with --no-deps
# - exact CPU Torch/TorchVision/TorchAudio matching versions
# - torchvision::nms validation
# - Transformers / Llama import validation
# - Perth / setuptools compatibility
# - known working dependency pins
# - FastAPI wrapper dependencies
#
# ======================================================================

import shutil
import socket
import subprocess
from pathlib import Path


# ======================================================================
# PATHS
# ======================================================================

ROOT = Path("/content/chatterbox_runtime")
PROJECT = ROOT / "Chatterbox-TTS-Server"
VENV = ROOT / "python310"
PYTHON = VENV / "bin/python"

SERVER_REPO = (
    "https://github.com/"
    "devnen/Chatterbox-TTS-Server.git"
)

CHATTERBOX_REPO = (
    "git+https://github.com/"
    "devnen/chatterbox-v2.git@master"
)


# ======================================================================
# HELPERS
# ======================================================================

def banner(text):
    print()
    print("=" * 78)
    print(text)
    print("=" * 78)


def run(cmd, check=True, cwd=None):
    cmd = [str(x) for x in cmd]

    print("\n>", " ".join(cmd))

    return subprocess.run(
        cmd,
        check=check,
        cwd=cwd,
        text=True
    )


def run_capture(cmd):
    return subprocess.run(
        [str(x) for x in cmd],
        capture_output=True,
        text=True
    )


# ======================================================================
# ENVIRONMENT TEST
# ======================================================================

def test_environment():

    if not PYTHON.exists():
        return False, "Python 3.10 environment does not exist."

    if not (PROJECT / "server.py").exists():
        return False, "Chatterbox server repository does not exist."

    script = r'''
import sys

print("Python:", sys.version)

import torch
import torchvision
import torchaudio

print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("TorchAudio:", torchaudio.__version__)

# Exact failure we previously encountered:
from torchvision.ops import nms

boxes = torch.tensor([
    [0.0, 0.0, 10.0, 10.0],
    [1.0, 1.0, 9.0, 9.0]
])

scores = torch.tensor([
    0.90,
    0.80
])

result = nms(
    boxes,
    scores,
    0.5
)

print("torchvision::nms test:", result.tolist())

import chatterbox
import s3tokenizer
import onnx
import safetensors
import diffusers

from transformers import LlamaModel

import fastapi
import uvicorn
import multipart

print("ALL_IMPORTS_OK")
'''

    result = run_capture(
        [
            PYTHON,
            "-c",
            script
        ]
    )

    if result.returncode != 0:
        return False, (
            result.stdout
            + "\n"
            + result.stderr
        )

    if "ALL_IMPORTS_OK" not in result.stdout:
        return False, result.stdout

    return True, result.stdout


# ======================================================================
# START
# ======================================================================

banner(
    "YOUTUBE BLUE — CHATTERBOX INSTALL / REPAIR"
)


# ======================================================================
# 1. UV
# ======================================================================

print("\n[1/8] Checking uv...")

UV = shutil.which("uv")

if not UV:

    run([
        "bash",
        "-lc",
        (
            "curl -LsSf "
            "https://astral.sh/uv/install.sh | sh"
        )
    ])

    UV = (
        shutil.which("uv")
        or
        "/root/.local/bin/uv"
    )

if not Path(UV).exists():
    raise RuntimeError(
        "uv could not be installed."
    )

print("✅ uv ready")


# ======================================================================
# 2. PYTHON 3.10
# ======================================================================

print("\n[2/8] Checking Python 3.10 environment...")

ROOT.mkdir(
    parents=True,
    exist_ok=True
)

if not PYTHON.exists():

    run([
        UV,
        "python",
        "install",
        "3.10"
    ])

    run([
        UV,
        "venv",
        "--python",
        "3.10",
        "--seed",
        VENV
    ])

run([
    PYTHON,
    "--version"
])

print("✅ Python 3.10 ready")


# ======================================================================
# 3. SERVER REPOSITORY
# ======================================================================

print("\n[3/8] Checking Chatterbox server repository...")

if not (PROJECT / "server.py").exists():

    if PROJECT.exists():
        shutil.rmtree(PROJECT)

    run([
        "git",
        "clone",
        SERVER_REPO,
        PROJECT
    ])

print("✅ Server repository ready")


# ======================================================================
# 4. SEE IF ENVIRONMENT IS ALREADY PERFECT
# ======================================================================

print("\n[4/8] Testing current environment...")

healthy, details = test_environment()

if healthy:

    print(details)

    banner(
        "✅ INSTALL ALREADY HEALTHY — NOTHING REBUILT"
    )

    print(
        "Cell 1 is complete.\n"
        "Now run Cell 2."
    )

else:

    print(
        "Environment needs installation or repair."
    )

    print("\nCurrent test result:")
    print(details[-5000:])


    # ==================================================================
    # 5. INSTALL SERVER DEPENDENCIES
    # ==================================================================

    print(
        "\n[5/8] Installing server dependencies..."
    )

    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pip",
        "wheel",
        "setuptools<81"
    ])

    # The upstream server's CPU requirements.
    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "-r",
        PROJECT / "requirements.txt"
    ])


    # ==================================================================
    # 6. INSTALL CHATTERBOX-V2 CORRECTLY
    # ==================================================================

    print(
        "\n[6/8] Installing Chatterbox engine..."
    )

    # IMPORTANT:
    # We do NOT pip install -e the server repository.
    #
    # chatterbox-v2 is the actual installable TTS package.
    #
    # --no-deps prevents pip from replacing our Torch stack.
    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--no-deps",
        CHATTERBOX_REPO,
        "s3tokenizer==0.3.0",
        "onnx==1.16.0"
    ])


    # ==================================================================
    # 7. APPLY KNOWN WORKING COMPATIBILITY PINS
    # ==================================================================

    print(
        "\n[7/8] Applying compatibility pins..."
    )

    # --no-deps here is deliberate.
    # The requirements were already installed above.
    #
    # This prevents these repairs from unexpectedly replacing Torch.
    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",

        "numpy==1.25.2",
        "safetensors==0.5.3",
        "transformers==4.46.3",
        "diffusers==0.29.0",
        "s3tokenizer==0.3.0",
        "onnx==1.16.0",
        "protobuf==3.20.3",
        "resemble-perth==1.0.1",
        "setuptools<81"
    ])

    # Make sure wrapper dependencies exist.
    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",

        "fastapi<0.116",
        "uvicorn[standard]",
        "python-multipart",
        "requests",
        "PyYAML"
    ])


    # ==================================================================
    # 8. TORCH STACK — ALWAYS INSTALLED LAST
    # ==================================================================

    print(
        "\n[8/8] Repairing exact CPU Torch stack..."
    )

    # Completely remove potentially mismatched wheels first.
    subprocess.run(
        [
            str(PYTHON),
            "-m",
            "pip",
            "uninstall",
            "-y",
            "torch",
            "torchvision",
            "torchaudio"
        ],
        check=False
    )

    # Matching trio:
    #
    # torch       2.5.1
    # torchvision 0.20.1
    # torchaudio  2.5.1
    #
    # Explicit +cpu avoids mixed PyPI wheels.
    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",

        "torch==2.5.1+cpu",
        "torchvision==0.20.1+cpu",
        "torchaudio==2.5.1+cpu"
    ])

    # Perth compatibility.
    run([
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "setuptools<81"
    ])


    # ==================================================================
    # FINAL VERIFICATION
    # ==================================================================

    banner(
        "FINAL ENVIRONMENT VERIFICATION"
    )

    healthy, details = test_environment()

    print(details)

    if not healthy:

        raise RuntimeError(
            "\n"
            "Environment repair did not pass the final verification.\n"
            "Do NOT run Cell 2 yet."
        )

    banner(
        "✅ CELL 1 COMPLETE — CHATTERBOX ENVIRONMENT READY"
    )

    print(
        "The exact TorchVision NMS test passed.\n"
        "Chatterbox imported successfully.\n"
        "Now run Cell 2."
    )


YOUTUBE BLUE — CHATTERBOX INSTALL / REPAIR

[1/8] Checking uv...
✅ uv ready

[2/8] Checking Python 3.10 environment...

> /content/chatterbox_runtime/python310/bin/python --version
✅ Python 3.10 ready

[3/8] Checking Chatterbox server repository...
✅ Server repository ready

[4/8] Testing current environment...
Environment needs installation or repair.

Current test result:
Python: 3.10.21 (main, Sep  1 2026, 14:16:49) [Clang 22.1.3 ]

Traceback (most recent call last):
  File "<string>", line 7, in <module>
  File "/content/chatterbox_runtime/python310/lib/python3.10/site-packages/torchvision/__init__.py", line 10, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
  File "/content/chatterbox_runtime/python310/lib/python3.10/site-packages/torchvision/_meta_registrations.py", line 164, in <module>
    def meta_nms(dets, scores, iou_threshold):
  File "/content/chatterbox_runtime/python310/lib/python3.10/site-packages

In [ ]:
# ======================================================================
# YOUTUBE BLUE — CELL 2
# SELF-HEALING MULTI-MODEL BACKEND
# + VOICE SHARING
# + CLOUDFLARE
# + GITHUB CONFIG
# + WATCHDOG
# ======================================================================

import os
import re
import json
import time
import base64
import socket
import shutil
import requests
import subprocess

from pathlib import Path
from datetime import datetime, timezone


# ======================================================================
# GLOBAL PATHS
# ======================================================================

ROOT = Path("/content/chatterbox_runtime")
PROJECT = ROOT / "Chatterbox-TTS-Server"
VENV = ROOT / "python310"
PYTHON = VENV / "bin/python"

CONFIG_PATH = PROJECT / "config.yaml"

CHATTERBOX_PORT = 8004
JOB_API_PORT = 8010

SERVER_LOG = Path("/content/chatterbox_server.log")
JOB_LOG = Path("/content/chatterbox_job_api.log")
TUNNEL_LOG = Path("/content/cloudflared.log")
JOB_APP = Path("/content/chatterbox_job_api.py")

CLOUDFLARED = Path("/usr/local/bin/cloudflared")


# ======================================================================
# REPOSITORIES
# ======================================================================

SERVER_REPO = (
    "https://github.com/"
    "devnen/Chatterbox-TTS-Server.git"
)

CHATTERBOX_ENGINE_REPO = (
    "git+https://github.com/"
    "devnen/chatterbox-v2.git@master"
)


# ======================================================================
# GITHUB
# ======================================================================

GITHUB_OWNER = "Zayaiken21"
GITHUB_REPO = "Youtube-Blue"
GITHUB_BRANCH = "main"
GITHUB_FILE = "backend-config.json"

PAGES_CONFIG = (
    "https://zayaiken21.github.io/"
    "Youtube-Blue/backend-config.json"
)

FRONTEND_URL = (
    "https://zayaiken21.github.io/"
    "Youtube-Blue/"
)


# ======================================================================
# RUNTIME STATE
# ======================================================================

SERVER_PROCESS = None
JOB_PROCESS = None
TUNNEL_PROCESS = None

MODEL_INFO = None
PUBLIC_URL = None
LAST_PUBLISHED_MODEL = None


# ======================================================================
# DISPLAY HELPERS
# ======================================================================

def banner(text):

    print()
    print("=" * 78)
    print(text)
    print("=" * 78)


def tail(path, count=180):

    path = Path(path)

    if not path.exists():
        return "(no log)"

    lines = path.read_text(
        errors="ignore"
    ).splitlines()

    return "\n".join(
        lines[-count:]
    )


# ======================================================================
# PROCESS HELPERS
# ======================================================================

def kill_pattern(pattern):

    subprocess.run(
        [
            "pkill",
            "-9",
            "-f",
            pattern
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )


def kill_port(port):

    subprocess.run(
        [
            "bash",
            "-lc",
            f"fuser -k -n tcp {port} 2>/dev/null || true"
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )


def port_open(port):

    sock = socket.socket()
    sock.settimeout(1)

    try:

        return (
            sock.connect_ex(
                ("127.0.0.1", port)
            )
            ==
            0
        )

    finally:

        sock.close()


# ======================================================================
# INSTALL COMMAND
# ======================================================================

def install_run(command, check=True):

    command = [
        str(x)
        for x in command
    ]

    print()
    print(
        ">",
        " ".join(command)
    )

    return subprocess.run(
        command,
        check=check,
        text=True
    )


# ======================================================================
# ENVIRONMENT TEST
# ======================================================================

def environment_test():

    if not PYTHON.exists():

        return (
            False,
            "Python 3.10 environment is missing."
        )


    if not (
        PROJECT /
        "server.py"
    ).exists():

        return (
            False,
            "Chatterbox server repository is missing."
        )


    test_script = r'''
import sys

print("Python:", sys.version)

import torch
import torchvision
import torchaudio

print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("TorchAudio:", torchaudio.__version__)

from torchvision.ops import nms

boxes = torch.tensor([
    [0.0, 0.0, 10.0, 10.0],
    [1.0, 1.0, 9.0, 9.0]
])

scores = torch.tensor([
    0.90,
    0.80
])

result = nms(
    boxes,
    scores,
    0.5
)

print(
    "torchvision::nms:",
    result.tolist()
)

import chatterbox
import s3tokenizer
import onnx
import safetensors
import diffusers

from transformers import LlamaModel

import fastapi
import uvicorn
import multipart
import requests
import yaml

print("SELF_HEAL_TEST_OK")
'''


    result = subprocess.run(
        [
            str(PYTHON),
            "-c",
            test_script
        ],
        capture_output=True,
        text=True
    )


    combined = (
        result.stdout
        +
        "\n"
        +
        result.stderr
    )


    if result.returncode != 0:

        return (
            False,
            combined
        )


    if (
        "SELF_HEAL_TEST_OK"
        not in
        result.stdout
    ):

        return (
            False,
            combined
        )


    return (
        True,
        result.stdout
    )


# ======================================================================
# SELF-HEAL INSTALL
# ======================================================================

def ensure_uv():

    uv = shutil.which("uv")

    if uv:
        return uv


    print(
        "uv missing. Installing..."
    )


    install_run(
        [
            "bash",
            "-lc",
            (
                "curl -LsSf "
                "https://astral.sh/uv/install.sh "
                "| sh"
            )
        ]
    )


    uv = (
        shutil.which("uv")
        or
        "/root/.local/bin/uv"
    )


    if not Path(uv).exists():

        raise RuntimeError(
            "uv installation failed."
        )


    return uv


def ensure_python_environment():

    if PYTHON.exists():
        return


    print(
        "\nPython 3.10 environment missing."
    )

    print(
        "Creating automatically..."
    )


    uv = ensure_uv()


    ROOT.mkdir(
        parents=True,
        exist_ok=True
    )


    install_run(
        [
            uv,
            "python",
            "install",
            "3.10"
        ]
    )


    install_run(
        [
            uv,
            "venv",
            "--python",
            "3.10",
            "--seed",
            VENV
        ]
    )


    if not PYTHON.exists():

        raise RuntimeError(
            "Python 3.10 environment creation failed."
        )


def ensure_server_repository():

    if (
        PROJECT /
        "server.py"
    ).exists():

        return


    print(
        "\nChatterbox server missing."
    )

    print(
        "Cloning automatically..."
    )


    ROOT.mkdir(
        parents=True,
        exist_ok=True
    )


    if PROJECT.exists():

        shutil.rmtree(
            PROJECT,
            ignore_errors=True
        )


    install_run(
        [
            "git",
            "clone",
            SERVER_REPO,
            PROJECT
        ]
    )


    if not (
        PROJECT /
        "server.py"
    ).exists():

        raise RuntimeError(
            "Chatterbox server clone failed."
        )


def repair_environment():

    banner(
        "⚠️ AUTOMATIC CHATTERBOX ENVIRONMENT REPAIR"
    )


    ensure_python_environment()
    ensure_server_repository()


    install_run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "pip",
            "wheel",
            "setuptools<81"
        ]
    )


    install_run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "-r",
            PROJECT /
            "requirements.txt"
        ]
    )


    install_run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-deps",
            CHATTERBOX_ENGINE_REPO,
            "s3tokenizer==0.3.0",
            "onnx==1.16.0"
        ]
    )


    install_run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--force-reinstall",
            "--no-deps",

            "numpy==1.25.2",
            "safetensors==0.5.3",
            "transformers==4.46.3",
            "diffusers==0.29.0",
            "s3tokenizer==0.3.0",
            "onnx==1.16.0",
            "protobuf==3.20.3",
            "resemble-perth==1.0.1",
            "setuptools<81"
        ]
    )


    install_run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",

            "fastapi<0.116",
            "uvicorn[standard]",
            "python-multipart",
            "requests",
            "PyYAML"
        ]
    )


    subprocess.run(
        [
            str(PYTHON),
            "-m",
            "pip",
            "uninstall",
            "-y",
            "torch",
            "torchvision",
            "torchaudio"
        ],
        check=False
    )


    install_run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--force-reinstall",
            "--no-deps",
            "--index-url",
            "https://download.pytorch.org/whl/cpu",

            "torch==2.5.1+cpu",
            "torchvision==0.20.1+cpu",
            "torchaudio==2.5.1+cpu"
        ]
    )


    install_run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "setuptools<81"
        ]
    )


    healthy, details = environment_test()

    print(details)


    if not healthy:

        raise RuntimeError(
            "Automatic environment repair failed verification."
        )


    banner(
        "✅ ENVIRONMENT AUTOMATICALLY REPAIRED"
    )


# ======================================================================
# START
# ======================================================================

banner(
    "YOUTUBE BLUE — SELF-HEALING MULTI-MODEL BACKEND"
)


print(
    "\n[1/10] Verifying environment..."
)


healthy, details = environment_test()


if healthy:

    print(
        "✅ Installed environment healthy."
    )


else:

    print(
        "⚠️ Environment missing or unhealthy."
    )

    print(
        details[-4000:]
    )


    repair_environment()


    healthy, details = environment_test()


    if not healthy:

        print(details)

        raise RuntimeError(
            "Environment self-repair failed."
        )


    print(
        "✅ Environment repaired and verified."
    )


# ======================================================================
# GITHUB TOKEN
# ======================================================================

print(
    "\n[2/10] Checking Github access..."
)


from google.colab import userdata


GITHUB_TOKEN = (
    userdata.get(
        "GITHUB_TOKEN"
    )
    or
    ""
).strip()


if not GITHUB_TOKEN:

    raise RuntimeError(
        "Missing Colab Secret: GITHUB_TOKEN"
    )


GH_HEADERS = {

    "Authorization":
        f"Bearer {GITHUB_TOKEN}",

    "Accept":
        "application/vnd.github+json",

    "X-GitHub-Api-Version":
        "2022-11-28"
}


repo_test = requests.get(
    (
        "https://api.github.com/repos/"
        f"{GITHUB_OWNER}/"
        f"{GITHUB_REPO}"
    ),
    headers=GH_HEADERS,
    timeout=20
)


print(
    "Github status:",
    repo_test.status_code
)


if repo_test.status_code != 200:

    print(
        repo_test.text[:1500]
    )

    raise RuntimeError(
        "GITHUB_TOKEN cannot access Youtube-Blue."
    )


print(
    "✅ Github access confirmed."
)


# ======================================================================
# CHATTERBOX CONFIG
# ======================================================================

print(
    "\n[3/10] Preparing Chatterbox configuration..."
)


import yaml


with open(
    CONFIG_PATH,
    "r"
) as f:

    config = (
        yaml.safe_load(f)
        or
        {}
    )


config.setdefault(
    "server",
    {}
)

config["server"]["host"] = "0.0.0.0"
config["server"]["port"] = CHATTERBOX_PORT


config.setdefault(
    "model",
    {}
)

config["model"]["repo_id"] = "chatterbox-turbo"


config.pop(
    "model_selector",
    None
)


config.setdefault(
    "tts_engine",
    {}
)

config[
    "tts_engine"
][
    "device"
] = "cpu"


config.setdefault(
    "generation_defaults",
    {}
)

config[
    "generation_defaults"
][
    "temperature"
] = 0.75

config[
    "generation_defaults"
][
    "exaggeration"
] = 0.5

config[
    "generation_defaults"
][
    "cfg_weight"
] = 0.5

config[
    "generation_defaults"
][
    "speed_factor"
] = 1.0


with open(
    CONFIG_PATH,
    "w"
) as f:

    yaml.safe_dump(
        config,
        f,
        sort_keys=False
    )


print(
    "Startup model: chatterbox-turbo"
)

print(
    "Device: cpu"
)


# ======================================================================
# CLEAN OLD PROCESSES
# ======================================================================

print(
    "\n[4/10] Clearing stale services..."
)


kill_pattern(
    "cloudflared"
)

kill_pattern(
    "chatterbox_job_api"
)

kill_port(
    JOB_API_PORT
)

kill_pattern(
    "server.py"
)

kill_port(
    CHATTERBOX_PORT
)


time.sleep(3)


if port_open(
    CHATTERBOX_PORT
):

    raise RuntimeError(
        "Port 8004 is still occupied."
    )


if port_open(
    JOB_API_PORT
):

    raise RuntimeError(
        "Port 8010 is still occupied."
    )


print(
    "✅ Old processes cleared."
)


# ======================================================================
# CHATTERBOX STARTER
# ======================================================================

def start_chatterbox():

    global SERVER_PROCESS
    global MODEL_INFO


    print(
        "\nStarting Chatterbox..."
    )


    kill_pattern(
        "server.py"
    )

    kill_port(
        CHATTERBOX_PORT
    )

    time.sleep(2)


    env = os.environ.copy()

    env[
        "OMP_NUM_THREADS"
    ] = "2"

    env[
        "MKL_NUM_THREADS"
    ] = "2"

    env[
        "OPENBLAS_NUM_THREADS"
    ] = "2"

    env[
        "NUMEXPR_NUM_THREADS"
    ] = "2"

    env[
        "MALLOC_ARENA_MAX"
    ] = "2"


    log_handle = open(
        SERVER_LOG,
        "w"
    )


    SERVER_PROCESS = subprocess.Popen(
        [
            str(PYTHON),
            "server.py"
        ],
        cwd=str(PROJECT),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=env
    )


    print(
        "Chatterbox PID:",
        SERVER_PROCESS.pid
    )


    MODEL_INFO = None


    for attempt in range(
        450
    ):

        code = SERVER_PROCESS.poll()


        if code is not None:

            print(
                tail(
                    SERVER_LOG,
                    220
                )
            )

            raise RuntimeError(
                "Chatterbox process exited."
            )


        try:

            response = requests.get(
                (
                    "http://127.0.0.1:"
                    f"{CHATTERBOX_PORT}"
                    "/api/model-info"
                ),
                timeout=5
            )


            if response.status_code == 200:

                info = response.json()


                if (
                    info.get(
                        "loaded"
                    )
                    is True
                ):

                    MODEL_INFO = info

                    break


        except Exception:

            pass


        if attempt % 10 == 0:

            print(
                "Loading model...",
                f"{attempt * 2}s"
            )


        time.sleep(2)


    if not MODEL_INFO:

        raise RuntimeError(
            "Chatterbox did not become ready."
        )


    print(
        json.dumps(
            MODEL_INFO,
            indent=2
        )
    )


    print(
        "✅ CHATTERBOX READY:",
        MODEL_INFO.get(
            "type"
        )
    )


print(
    "\n[5/10] Starting Chatterbox..."
)

start_chatterbox()


# ======================================================================
# FASTAPI WRAPPER
# ======================================================================

print(
    "\n[6/10] Building Youtube Blue API..."
)


WRAPPER_CODE = r'''
import os
import io
import re
import uuid
import time
import wave
import base64
import hashlib
import threading

from pathlib import Path
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List
from urllib.parse import quote

import requests

from fastapi import (
    FastAPI,
    HTTPException,
    UploadFile,
    File,
    Form
)

from fastapi.middleware.cors import (
    CORSMiddleware
)

from fastapi.responses import (
    FileResponse,
    Response
)

from pydantic import (
    BaseModel,
    Field
)


# ==================================================================
# CORE CONFIG
# ==================================================================

CHATTERBOX = (
    "http://127.0.0.1:8004"
)

OUTPUT_DIR = Path(
    "/content/chatterbox_jobs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================
# GITHUB VOICE SHARE CONFIG
# ==================================================================
#
# IMPORTANT:
# The literal secret is NEVER written into this file.
# It enters the uvicorn process through its environment.
#
# Keep only one literal GITHUB_TOKEN reference so:
#
# grep -c GITHUB_TOKEN chatterbox_job_api.py
#
# reports 1.
# ==================================================================

GH_TOKEN = os.environ["GITHUB_TOKEN"]

GH_OWNER = "Zayaiken21"
GH_REPO = "Youtube-Blue"
GH_BRANCH = "main"

VOICE_FOLDER = "Voices/Our Voices"

GH_API = (
    "https://api.github.com"
)

GH_HEADERS = {

    "Authorization":
        "Bearer " + GH_TOKEN,

    "Accept":
        "application/vnd.github+json",

    "X-GitHub-Api-Version":
        "2022-11-28"
}


MAX_WAV_BYTES = (
    15 *
    1024 *
    1024
)

MAX_MP3_BYTES = (
    2 *
    1024 *
    1024
)


# ==================================================================
# FASTAPI
# ==================================================================

app = FastAPI(
    title="Youtube Blue Chatterbox API",
    version="3.3"
)


app.add_middleware(
    CORSMiddleware,

    allow_origins=[
        "https://zayaiken21.github.io",
        "http://localhost",
        "http://127.0.0.1"
    ],

    allow_credentials=False,

    allow_methods=["*"],

    allow_headers=["*"]
)


# ==================================================================
# RUNTIME STATE
# ==================================================================

operation_executor = ThreadPoolExecutor(
    max_workers=1
)

state_lock = threading.Lock()

jobs = {}

model_switch_jobs = {}

model_switching = False


MODEL_MAP = {

    "turbo":
        "chatterbox-turbo",

    "original":
        "chatterbox",

    "multilingual":
        "chatterbox-multilingual"
}


# ==================================================================
# SHARE VOICE STATE
# ==================================================================

share_lock = threading.Lock()

share_success_times = deque()

share_idempotency = {}


# ==================================================================
# REQUEST MODELS
# ==================================================================

class JobRequest(
    BaseModel
):

    text: str

    voice_mode: str = "predefined"

    predefined_voice_id: Optional[str] = None

    reference_audio_filename: Optional[str] = None

    output_format: str = "wav"

    split_text: bool = True

    chunk_size: int = Field(
        default=400,
        ge=50,
        le=500
    )

    temperature: Optional[float] = Field(
        default=0.75,
        ge=0.0,
        le=1.5
    )

    exaggeration: Optional[float] = Field(
        default=None,
        ge=0.25,
        le=2.0
    )

    cfg_weight: Optional[float] = Field(
        default=None,
        ge=0.2,
        le=1.0
    )

    seed: Optional[int] = Field(
        default=0,
        ge=0
    )

    speed_factor: Optional[float] = Field(
        default=1.0,
        ge=0.25,
        le=4.0
    )

    language: Optional[str] = "en"


class ModelSwitchRequest(
    BaseModel
):

    model: str


# ==================================================================
# COMMON HELPERS
# ==================================================================

def model_dump_compat(
    model
):

    if hasattr(
        model,
        "model_dump"
    ):

        return model.model_dump(
            exclude_none=True
        )


    return model.dict(
        exclude_none=True
    )


def chatterbox_model_info(
    timeout=10
):

    response = requests.get(
        CHATTERBOX +
        "/api/model-info",
        timeout=timeout
    )

    response.raise_for_status()

    return response.json()


def generation_active():

    return any(

        job.get(
            "status"
        )
        in
        (
            "queued",
            "running"
        )

        for job in
        jobs.values()
    )


def clean_payload_for_model(
    payload,
    model_type
):

    payload = dict(
        payload
    )


    if model_type == "turbo":

        payload.pop(
            "exaggeration",
            None
        )

        payload.pop(
            "cfg_weight",
            None
        )

        payload[
            "language"
        ] = "en"


    elif model_type == "original":

        payload[
            "language"
        ] = "en"


    elif model_type == "multilingual":

        payload.pop(
            "cfg_weight",
            None
        )


    return payload


# ==================================================================
# VOICE SHARE HELPERS
# ==================================================================

def clean_voice_name(
    value
):

    value = (
        value
        or
        ""
    )


    # Keep:
    #
    # letters
    # digits
    # spaces
    # apostrophe
    # hyphen
    #
    cleaned_chars = []


    for char in value:

        if (
            char.isalnum()
            or
            char in (
                " ",
                "'",
                "-"
            )
        ):

            cleaned_chars.append(
                char
            )


    cleaned = "".join(
        cleaned_chars
    )


    cleaned = re.sub(
        r"\s+",
        " ",
        cleaned
    ).strip()


    cleaned = (
        cleaned[:40]
        .strip()
    )


    return cleaned


def git_blob_sha(
    data
):

    header = (
        b"blob "
        +
        str(
            len(data)
        ).encode(
            "ascii"
        )
        +
        b"\0"
    )


    return hashlib.sha1(
        header +
        data
    ).hexdigest()


def encoded_repo_path(
    repo_path
):

    # Keep "/" separators while encoding
    # spaces and filename characters.
    return quote(
        repo_path,
        safe="/"
    )


def github_content_url(
    repo_path
):

    return (
        GH_API
        +
        "/repos/"
        +
        GH_OWNER
        +
        "/"
        +
        GH_REPO
        +
        "/contents/"
        +
        encoded_repo_path(
            repo_path
        )
    )


def prune_share_rate_limit():

    cutoff = (
        time.time()
        -
        3600
    )


    while (
        share_success_times
        and
        share_success_times[0]
        <=
        cutoff
    ):

        share_success_times.popleft()


def log_share(
    message
):

    # Never include credentials.
    print(
        "[share-voice] "
        +
        str(
            message
        ),
        flush=True
    )


def validate_shared_audio(
    filename,
    data
):

    suffix = (
        Path(
            filename
            or
            ""
        )
        .suffix
        .lower()
    )


    if suffix not in (
        ".wav",
        ".mp3"
    ):

        raise HTTPException(
            status_code=422,
            detail=(
                "Only WAV and MP3 voice files "
                "can be shared."
            )
        )


    if suffix == ".wav":

        if len(data) > MAX_WAV_BYTES:

            raise HTTPException(
                status_code=413,
                detail=(
                    "WAV file is too large. "
                    "Maximum size is 15 MB."
                )
            )


        try:

            with wave.open(
                io.BytesIO(
                    data
                ),
                "rb"
            ) as wav_file:

                frame_rate = (
                    wav_file.getframerate()
                )

                frame_count = (
                    wav_file.getnframes()
                )


                if frame_rate <= 0:

                    raise ValueError(
                        "Invalid WAV sample rate."
                    )


                duration = (
                    frame_count
                    /
                    float(
                        frame_rate
                    )
                )


        except Exception:

            raise HTTPException(
                status_code=422,
                detail=(
                    "The uploaded file is not "
                    "a valid WAV audio file."
                )
            )


        if duration < 1.0:

            raise HTTPException(
                status_code=422,
                detail=(
                    "Voice audio must be at least "
                    "1 second long."
                )
            )


        if duration > 30.5:

            raise HTTPException(
                status_code=422,
                detail=(
                    "Voice audio must be 30 seconds "
                    "or shorter."
                )
            )


    else:

        if len(data) > MAX_MP3_BYTES:

            raise HTTPException(
                status_code=413,
                detail=(
                    "MP3 file is too large. "
                    "Maximum size is 2 MB."
                )
            )


        valid_mp3 = (

            data.startswith(
                b"ID3"
            )

            or

            data[:2]
            in
            (
                b"\xff\xfb",
                b"\xff\xf3",
                b"\xff\xf2"
            )
        )


        if not valid_mp3:

            raise HTTPException(
                status_code=422,
                detail=(
                    "The uploaded file is not "
                    "a valid MP3 audio file."
                )
            )


    return suffix


# ==================================================================
# ROOT
# ==================================================================

@app.get("/")
def root():

    return {

        "ok":
            True,

        "service":
            "youtube-blue-chatterbox-api",

        "version":
            "3.3",

        "health":
            "/health",

        "model_info":
            "/model-info",

        "models":
            "/models",

        "voice_share":
            "/share-voice",

        "docs":
            "/docs"
    }


# ==================================================================
# HEALTH
# ==================================================================

@app.get("/health")
def health():

    return {

        "ok":
            True,

        "service":
            "youtube-blue-chatterbox-job-api"
    }


# ==================================================================
# STATE
# ==================================================================

@app.get("/state")
def backend_state():

    with state_lock:

        return {

            "generation_active":
                generation_active(),

            "model_switching":
                model_switching
        }


# ==================================================================
# MODEL INFO
# ==================================================================

@app.get("/model-info")
def model_info():

    try:

        response = requests.get(
            CHATTERBOX +
            "/api/model-info",
            timeout=10
        )


        return Response(
            content=
                response.content,

            status_code=
                response.status_code,

            media_type=
                response.headers.get(
                    "content-type",
                    "application/json"
                )
        )


    except Exception as exc:

        return {

            "loaded":
                False,

            "busy_or_unreachable":
                True,

            "error":
                str(
                    exc
                )
        }


# ==================================================================
# MODELS
# ==================================================================

@app.get("/models")
def available_models():

    try:

        info = chatterbox_model_info()


    except Exception as exc:

        raise HTTPException(
            status_code=502,
            detail=str(
                exc
            )
        )


    return {

        "active":
            info.get(
                "type"
            ),

        "models": [

            {
                "id": "turbo",
                "name": "Chatterbox Turbo",

                "available":
                    bool(
                        info.get(
                            "turbo_available_in_package",
                            False
                        )
                    ),

                "supports_exaggeration":
                    False,

                "supports_cfg_weight":
                    False,

                "supports_paralinguistic_tags":
                    True,

                "supports_multilingual":
                    False
            },

            {
                "id": "original",
                "name": "Original Chatterbox",
                "available": True,

                "supports_exaggeration":
                    True,

                "supports_cfg_weight":
                    True,

                "supports_paralinguistic_tags":
                    False,

                "supports_multilingual":
                    False
            },

            {
                "id": "multilingual",
                "name": "Chatterbox Multilingual",

                "available":
                    bool(
                        info.get(
                            "multilingual_available_in_package",
                            False
                        )
                    ),

                "supports_exaggeration":
                    True,

                "supports_cfg_weight":
                    False,

                "supports_paralinguistic_tags":
                    False,

                "supports_multilingual":
                    True
            }
        ]
    }


# ==================================================================
# PREDEFINED VOICES
# ==================================================================

@app.get("/get_predefined_voices")
def predefined_voices():

    try:

        response = requests.get(
            CHATTERBOX +
            "/get_predefined_voices",
            timeout=30
        )


        return Response(
            content=
                response.content,

            status_code=
                response.status_code,

            media_type=
                response.headers.get(
                    "content-type",
                    "application/json"
                )
        )


    except Exception as exc:

        raise HTTPException(
            status_code=502,
            detail=str(
                exc
            )
        )


# ==================================================================
# REFERENCE FILES
# ==================================================================

@app.get("/get_reference_files")
def reference_files():

    try:

        response = requests.get(
            CHATTERBOX +
            "/get_reference_files",
            timeout=30
        )


        return Response(
            content=
                response.content,

            status_code=
                response.status_code,

            media_type=
                response.headers.get(
                    "content-type",
                    "application/json"
                )
        )


    except Exception as exc:

        raise HTTPException(
            status_code=502,
            detail=str(
                exc
            )
        )


# ==================================================================
# REFERENCE UPLOAD
# ==================================================================

@app.post("/upload_reference")
async def upload_reference(
    files: List[UploadFile] = File(...)
):

    with state_lock:

        if model_switching:

            raise HTTPException(
                status_code=409,
                detail=(
                    "Cannot upload reference audio "
                    "while model switching."
                )
            )


    multipart = []


    for file in files:

        data = await file.read()


        multipart.append(
            (
                "files",

                (
                    file.filename,

                    data,

                    file.content_type
                    or
                    "application/octet-stream"
                )
            )
        )


    try:

        response = requests.post(
            CHATTERBOX +
            "/upload_reference",

            files=
                multipart,

            timeout=
                180
        )


        return Response(
            content=
                response.content,

            status_code=
                response.status_code,

            media_type=
                response.headers.get(
                    "content-type",
                    "application/json"
                )
        )


    except Exception as exc:

        raise HTTPException(
            status_code=502,
            detail=str(
                exc
            )
        )


# ==================================================================
# SHARE VOICE TO GITHUB
# ==================================================================

@app.post("/share-voice")
def share_voice(
    file: UploadFile = File(...),
    name: str = Form(...),
    client_id: str = Form(...)
):

    final_name_for_log = None


    try:

        cleaned_name = clean_voice_name(
            name
        )


        if not cleaned_name:

            raise HTTPException(
                status_code=422,
                detail=(
                    "Voice name is empty after cleaning."
                )
            )


        cleaned_client_id = (
            client_id
            or
            ""
        ).strip()


        if (
            not cleaned_client_id

            or

            not re.fullmatch(
                r"[A-Za-z0-9]+",
                cleaned_client_id
            )
        ):

            raise HTTPException(
                status_code=422,
                detail=(
                    "client_id must contain only "
                    "letters and digits."
                )
            )


        original_suffix = (
            Path(
                file.filename
                or
                ""
            )
            .suffix
            .lower()
        )


        if original_suffix == ".wav":

            read_limit = (
                MAX_WAV_BYTES
                +
                1
            )


        elif original_suffix == ".mp3":

            read_limit = (
                MAX_MP3_BYTES
                +
                1
            )


        else:

            # Read only a tiny amount because the
            # extension is already invalid.
            read_limit = 32


        data = file.file.read(
            read_limit
        )


        if not data:

            raise HTTPException(
                status_code=422,
                detail="Audio file is empty."
            )


        suffix = validate_shared_audio(
            file.filename,
            data
        )


        blob_sha = git_blob_sha(
            data
        )


        idem_key = (
            cleaned_client_id,
            blob_sha
        )


        # ----------------------------------------------------------
        # SERIALIZE SHARE OPERATIONS
        #
        # This lock only affects /share-voice.
        # TTS generation remains independent.
        # ----------------------------------------------------------

        with share_lock:

            # ------------------------------------------------------
            # IDEMPOTENCY
            # ------------------------------------------------------

            previous = (
                share_idempotency.get(
                    idem_key
                )
            )


            if previous:

                result = dict(
                    previous
                )

                # Acceptance contract:
                # exact retry becomes already_shared
                # without touching Github again.
                result[
                    "status"
                ] = "already_shared"


                log_share(
                    (
                        result[
                            "name"
                        ]
                        +
                        " already_shared"
                    )
                )


                return result


            # ------------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------------

            prune_share_rate_limit()


            if (
                len(
                    share_success_times
                )
                >=
                20
            ):

                raise HTTPException(
                    status_code=429,
                    detail=(
                        "Voice share limit reached. "
                        "Please try again later."
                    )
                )


            # ------------------------------------------------------
            # TRY NAME, NAME 2 ... NAME 50
            # ------------------------------------------------------

            for number in range(
                1,
                51
            ):

                if number == 1:

                    candidate_name = (
                        cleaned_name
                    )


                else:

                    candidate_name = (
                        cleaned_name
                        +
                        " "
                        +
                        str(
                            number
                        )
                    )


                final_name_for_log = (
                    candidate_name
                )


                repo_path = (
                    VOICE_FOLDER
                    +
                    "/"
                    +
                    candidate_name
                    +
                    suffix
                )


                content_url = (
                    github_content_url(
                        repo_path
                    )
                )


                # --------------------------------------------------
                # CHECK CURRENT NAME
                # --------------------------------------------------

                try:

                    existing = requests.get(
                        content_url,

                        headers=
                            GH_HEADERS,

                        params={
                            "ref":
                                GH_BRANCH
                        },

                        timeout=30
                    )


                except requests.RequestException as exc:

                    raise HTTPException(
                        status_code=502,
                        detail=(
                            "Github lookup failed: "
                            +
                            str(
                                exc
                            )
                        )
                    )


                # --------------------------------------------------
                # SAME AUDIO ALREADY THERE
                # --------------------------------------------------

                if existing.status_code == 200:

                    try:

                        existing_data = (
                            existing.json()
                        )

                    except Exception:

                        raise HTTPException(
                            status_code=502,
                            detail=(
                                "Github returned an invalid "
                                "response while checking the voice."
                            )
                        )


                    if (
                        existing_data.get(
                            "sha"
                        )
                        ==
                        blob_sha
                    ):

                        result = {

                            "ok":
                                True,

                            "status":
                                "already_shared",

                            "name":
                                candidate_name,

                            "path":
                                repo_path
                        }


                        share_idempotency[
                            idem_key
                        ] = dict(
                            result
                        )


                        share_success_times.append(
                            time.time()
                        )


                        log_share(
                            (
                                candidate_name
                                +
                                " already_shared"
                            )
                        )


                        return result


                    # Same filename, different audio.
                    # Continue to next suffix.
                    continue


                # --------------------------------------------------
                # NAME AVAILABLE
                # --------------------------------------------------

                if existing.status_code == 404:

                    put_body = {

                        "message":
                            (
                                "Share voice: "
                                +
                                candidate_name
                            ),

                        "content":
                            base64.b64encode(
                                data
                            ).decode(
                                "ascii"
                            ),

                        "branch":
                            GH_BRANCH
                    }


                    try:

                        created = requests.put(
                            content_url,

                            headers=
                                GH_HEADERS,

                            json=
                                put_body,

                            timeout=60
                        )


                    except requests.RequestException as exc:

                        raise HTTPException(
                            status_code=502,
                            detail=(
                                "Github upload failed: "
                                +
                                str(
                                    exc
                                )
                            )
                        )


                    # ----------------------------------------------
                    # CREATED
                    # ----------------------------------------------

                    if (
                        created.status_code
                        in
                        (
                            200,
                            201
                        )
                    ):

                        result = {

                            "ok":
                                True,

                            "status":
                                "published",

                            "name":
                                candidate_name,

                            "path":
                                repo_path
                        }


                        share_idempotency[
                            idem_key
                        ] = dict(
                            result
                        )


                        share_success_times.append(
                            time.time()
                        )


                        log_share(
                            (
                                candidate_name
                                +
                                " published"
                            )
                        )


                        return result


                    # ----------------------------------------------
                    # SOMEONE TOOK THAT NAME BETWEEN GET AND PUT
                    # ----------------------------------------------

                    if (
                        created.status_code
                        in
                        (
                            409,
                            422
                        )
                    ):

                        continue


                    raise HTTPException(
                        status_code=502,
                        detail=(
                            "Github upload failed with HTTP "
                            +
                            str(
                                created.status_code
                            )
                            +
                            "."
                        )
                    )


                # --------------------------------------------------
                # OTHER GITHUB ERROR
                # --------------------------------------------------

                raise HTTPException(
                    status_code=502,
                    detail=(
                        "Github lookup failed with HTTP "
                        +
                        str(
                            existing.status_code
                        )
                        +
                        "."
                    )
                )


            # All 50 candidates exhausted.
            raise HTTPException(
                status_code=502,
                detail=(
                    "Could not allocate a unique "
                    "voice filename."
                )
            )


    except HTTPException as exc:

        log_share(
            (
                (
                    final_name_for_log
                    or
                    clean_voice_name(
                        name
                    )
                    or
                    "unnamed"
                )
                +
                " error "
                +
                str(
                    exc.status_code
                )
                +
                ": "
                +
                str(
                    exc.detail
                )
            )
        )

        raise


    except Exception as exc:

        log_share(
            (
                (
                    final_name_for_log
                    or
                    "unnamed"
                )
                +
                " error 502: "
                +
                str(
                    exc
                )
            )
        )


        raise HTTPException(
            status_code=502,
            detail=(
                "Voice sharing failed because "
                "the Github request could not be completed."
            )
        )


# ==================================================================
# TTS GENERATION
# ==================================================================

def generate(
    job_id,
    payload
):

    job = jobs[
        job_id
    ]

    job[
        "status"
    ] = "running"


    try:

        info = chatterbox_model_info(
            timeout=15
        )


        model_type = (
            info.get(
                "type"
            )
        )


        payload = clean_payload_for_model(
            payload,
            model_type
        )


        job[
            "model"
        ] = model_type


        response = requests.post(
            CHATTERBOX +
            "/tts",

            json=
                payload,

            timeout=
                None
        )


        if not response.ok:

            job[
                "status"
            ] = "failed"


            job[
                "error"
            ] = (
                "Chatterbox HTTP "
                +
                str(
                    response.status_code
                )
                +
                ": "
                +
                response.text[:8000]
            )


            return


        content_type = (
            response.headers.get(
                "content-type",
                ""
            )
            .lower()
        )


        fmt = (
            payload.get(
                "output_format"
            )
            or
            "wav"
        ).lower()


        if "mpeg" in content_type:

            fmt = "mp3"


        elif "wav" in content_type:

            fmt = "wav"


        elif "ogg" in content_type:

            fmt = "ogg"


        if fmt not in (
            "wav",
            "mp3",
            "opus",
            "ogg"
        ):

            fmt = "wav"


        output_file = (
            OUTPUT_DIR /
            (
                job_id
                +
                "."
                +
                fmt
            )
        )


        output_file.write_bytes(
            response.content
        )


        job[
            "status"
        ] = "complete"


        job[
            "file"
        ] = str(
            output_file
        )


        job[
            "format"
        ] = fmt


    except Exception as exc:

        job[
            "status"
        ] = "failed"


        job[
            "error"
        ] = str(
            exc
        )


@app.post("/jobs")
def create_job(
    request: JobRequest
):

    global model_switching


    if not request.text.strip():

        raise HTTPException(
            status_code=400,
            detail="Text is required."
        )


    with state_lock:

        if model_switching:

            raise HTTPException(
                status_code=409,
                detail=(
                    "The speech model is currently switching."
                )
            )


    payload = model_dump_compat(
        request
    )


    voice_mode = (
        payload.get(
            "voice_mode",
            "predefined"
        )
    )


    if voice_mode == "clone":

        payload.pop(
            "predefined_voice_id",
            None
        )


        if not payload.get(
            "reference_audio_filename"
        ):

            raise HTTPException(
                status_code=400,
                detail=(
                    "Clone mode requires "
                    "reference_audio_filename."
                )
            )


    else:

        payload[
            "voice_mode"
        ] = "predefined"


        payload.pop(
            "reference_audio_filename",
            None
        )


        if not payload.get(
            "predefined_voice_id"
        ):

            raise HTTPException(
                status_code=400,
                detail=(
                    "Predefined mode requires "
                    "predefined_voice_id."
                )
            )


    job_id = str(
        uuid.uuid4()
    )


    jobs[
        job_id
    ] = {

        "job_id":
            job_id,

        "status":
            "queued",

        "error":
            None,

        "model":
            None
    }


    operation_executor.submit(
        generate,
        job_id,
        payload
    )


    return {

        "job_id":
            job_id,

        "status":
            "queued"
    }


@app.get("/jobs/{job_id}")
def job_status(
    job_id: str
):

    job = jobs.get(
        job_id
    )


    if not job:

        raise HTTPException(
            status_code=404,
            detail="Job not found."
        )


    return {

        "job_id":
            job_id,

        "status":
            job[
                "status"
            ],

        "model":
            job.get(
                "model"
            ),

        "format":
            job.get(
                "format"
            ),

        "error":
            job.get(
                "error"
            )
    }


@app.get("/jobs/{job_id}/audio")
def job_audio(
    job_id: str
):

    job = jobs.get(
        job_id
    )


    if not job:

        raise HTTPException(
            status_code=404,
            detail="Job not found."
        )


    if (
        job[
            "status"
        ]
        !=
        "complete"
    ):

        raise HTTPException(
            status_code=409,
            detail="Audio is not ready."
        )


    path = Path(
        job[
            "file"
        ]
    )


    if not path.exists():

        raise HTTPException(
            status_code=404,
            detail="Generated audio file is missing."
        )


    return FileResponse(
        path,
        filename=
            path.name
    )


# ==================================================================
# MODEL SWITCH
# ==================================================================

def perform_model_switch(
    switch_id,
    requested_model
):

    global model_switching


    switch = (
        model_switch_jobs[
            switch_id
        ]
    )


    switch[
        "status"
    ] = "switching"


    old_repo_id = None


    try:

        old_info = chatterbox_model_info(
            timeout=15
        )


        old_type = (
            old_info.get(
                "type"
            )
        )


        old_repo_id = (
            MODEL_MAP.get(
                old_type
            )
        )


        target_repo = (
            MODEL_MAP[
                requested_model
            ]
        )


        save_response = requests.post(
            CHATTERBOX +
            "/save_settings",

            json={
                "model": {
                    "repo_id":
                        target_repo
                }
            },

            timeout=30
        )


        if not save_response.ok:

            raise RuntimeError(
                (
                    "Could not save model setting: "
                    +
                    save_response.text[:3000]
                )
            )


        switch[
            "status"
        ] = "loading"


        reload_response = requests.post(
            CHATTERBOX +
            "/restart_server",

            timeout=None
        )


        if not reload_response.ok:

            raise RuntimeError(
                (
                    "Model hot-swap failed: "
                    +
                    reload_response.text[:5000]
                )
            )


        for _ in range(
            450
        ):

            try:

                info = chatterbox_model_info(
                    timeout=10
                )


                if (
                    info.get(
                        "loaded"
                    )
                    is True

                    and

                    info.get(
                        "type"
                    )
                    ==
                    requested_model
                ):

                    switch[
                        "status"
                    ] = "complete"


                    switch[
                        "model"
                    ] = requested_model


                    switch[
                        "model_info"
                    ] = info


                    return


            except Exception:

                pass


            time.sleep(2)


        raise RuntimeError(
            (
                "Model switch timed out before "
                "the requested model became ready."
            )
        )


    except Exception as exc:

        switch[
            "status"
        ] = "failed"


        switch[
            "error"
        ] = str(
            exc
        )


        if old_repo_id:

            try:

                requests.post(
                    CHATTERBOX +
                    "/save_settings",

                    json={
                        "model": {
                            "repo_id":
                                old_repo_id
                        }
                    },

                    timeout=30
                )


                requests.post(
                    CHATTERBOX +
                    "/restart_server",

                    timeout=None
                )


            except Exception:

                pass


    finally:

        with state_lock:

            model_switching = False


@app.post("/model-switch-jobs")
def create_model_switch_job(
    request: ModelSwitchRequest
):

    global model_switching


    requested = (
        request.model
        .strip()
        .lower()
    )


    if requested not in MODEL_MAP:

        raise HTTPException(
            status_code=400,
            detail=(
                "Invalid model. "
                "Use turbo, original, or multilingual."
            )
        )


    try:

        info = chatterbox_model_info(
            timeout=15
        )


    except Exception as exc:

        raise HTTPException(
            status_code=502,
            detail=str(
                exc
            )
        )


    if (
        requested == "turbo"

        and

        not info.get(
            "turbo_available_in_package",
            False
        )
    ):

        raise HTTPException(
            status_code=400,
            detail="Turbo model unavailable."
        )


    if (
        requested == "multilingual"

        and

        not info.get(
            "multilingual_available_in_package",
            False
        )
    ):

        raise HTTPException(
            status_code=400,
            detail="Multilingual model unavailable."
        )


    if (
        info.get(
            "loaded"
        )
        is True

        and

        info.get(
            "type"
        )
        ==
        requested
    ):

        return {

            "switch_id":
                None,

            "status":
                "complete",

            "model":
                requested,

            "already_active":
                True,

            "model_info":
                info
        }


    with state_lock:

        if model_switching:

            raise HTTPException(
                status_code=409,
                detail="A model switch is already running."
            )


        if generation_active():

            raise HTTPException(
                status_code=409,
                detail=(
                    "Cannot switch models while "
                    "speech generation is running."
                )
            )


        model_switching = True


    switch_id = str(
        uuid.uuid4()
    )


    model_switch_jobs[
        switch_id
    ] = {

        "switch_id":
            switch_id,

        "requested_model":
            requested,

        "status":
            "queued",

        "model":
            None,

        "model_info":
            None,

        "error":
            None
    }


    operation_executor.submit(
        perform_model_switch,
        switch_id,
        requested
    )


    return {

        "switch_id":
            switch_id,

        "status":
            "queued",

        "requested_model":
            requested
    }


@app.get(
    "/model-switch-jobs/{switch_id}"
)
def model_switch_status(
    switch_id: str
):

    switch = (
        model_switch_jobs.get(
            switch_id
        )
    )


    if not switch:

        raise HTTPException(
            status_code=404,
            detail="Model switch job not found."
        )


    return switch
'''


# ======================================================================
# WRITE + COMPILE WRAPPER
# ======================================================================

JOB_APP.write_text(
    WRAPPER_CODE,
    encoding="utf-8"
)


print(
    "✅ Wrapper file written."
)


print(
    "Checking wrapper syntax..."
)


syntax_test = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "py_compile",
        str(
            JOB_APP
        )
    ],
    capture_output=True,
    text=True
)


if syntax_test.returncode != 0:

    print(
        syntax_test.stderr
    )

    raise RuntimeError(
        "Generated wrapper failed syntax validation."
    )


print(
    "✅ Wrapper syntax valid."
)


# ======================================================================
# START WRAPPER
# ======================================================================

def start_wrapper():

    global JOB_PROCESS


    kill_pattern(
        "chatterbox_job_api"
    )


    kill_port(
        JOB_API_PORT
    )


    time.sleep(2)


    log_handle = open(
        JOB_LOG,
        "w"
    )


    # --------------------------------------------------------------
    # CRITICAL SECURITY:
    #
    # Pass token into child process only through its environment.
    #
    # It is NOT embedded into chatterbox_job_api.py.
    # --------------------------------------------------------------

    wrapper_env = os.environ.copy()

    wrapper_env[
        "GITHUB_TOKEN"
    ] = GITHUB_TOKEN


    JOB_PROCESS = subprocess.Popen(
        [
            str(PYTHON),

            "-m",

            "uvicorn",

            "chatterbox_job_api:app",

            "--host",

            "0.0.0.0",

            "--port",

            str(
                JOB_API_PORT
            )
        ],

        cwd="/content",

        stdout=
            log_handle,

        stderr=
            subprocess.STDOUT,

        env=
            wrapper_env
    )


    print(
        "Wrapper PID:",
        JOB_PROCESS.pid
    )


    for _ in range(
        60
    ):

        if (
            JOB_PROCESS.poll()
            is not None
        ):

            print(
                tail(
                    JOB_LOG,
                    220
                )
            )

            raise RuntimeError(
                "Youtube Blue wrapper exited."
            )


        try:

            response = requests.get(
                (
                    "http://127.0.0.1:"
                    f"{JOB_API_PORT}"
                    "/health"
                ),
                timeout=3
            )


            if (
                response.status_code
                ==
                200

                and

                response.json().get(
                    "ok"
                )
                is True
            ):

                print(
                    "✅ ASYNC WRAPPER READY"
                )

                return


        except Exception:

            pass


        time.sleep(1)


    raise RuntimeError(
        "Wrapper failed to become healthy."
    )


start_wrapper()


# ======================================================================
# LOCAL ROUTES
# ======================================================================

print(
    "\n[7/10] Testing local API..."
)


LOCAL_BASE = (
    "http://127.0.0.1:"
    f"{JOB_API_PORT}"
)


LOCAL_ROUTES = [

    "/health",

    "/model-info",

    "/models",

    "/state",

    "/get_predefined_voices",

    "/get_reference_files"
]


for route in LOCAL_ROUTES:

    response = requests.get(
        LOCAL_BASE +
        route,
        timeout=30
    )


    print(
        route,
        "→",
        response.status_code
    )


    if response.status_code != 200:

        print(
            response.text[:2500]
        )

        raise RuntimeError(
            "Local route failed: "
            + route
        )


# Explicitly verify the new route exists.
share_options = requests.options(
    LOCAL_BASE +
    "/share-voice",

    headers={
        "Origin":
            "https://zayaiken21.github.io",

        "Access-Control-Request-Method":
            "POST"
    },

    timeout=10
)


print(
    "/share-voice OPTIONS →",
    share_options.status_code
)


if share_options.status_code not in (
    200,
    204
):

    raise RuntimeError(
        "/share-voice CORS preflight failed."
    )


print(
    "✅ Local API fully healthy."
)


# ======================================================================
# CLOUDFLARE
# ======================================================================

print(
    "\n[8/10] Preparing Cloudflare..."
)


if not CLOUDFLARED.exists():

    subprocess.run(
        [
            "wget",
            "-q",

            (
                "https://github.com/"
                "cloudflare/cloudflared/"
                "releases/latest/download/"
                "cloudflared-linux-amd64"
            ),

            "-O",

            str(
                CLOUDFLARED
            )
        ],

        check=True
    )


    subprocess.run(
        [
            "chmod",
            "+x",
            str(
                CLOUDFLARED
            )
        ],

        check=True
    )


print(
    "✅ cloudflared ready."
)


# ======================================================================
# TUNNEL
# ======================================================================

def start_tunnel():

    global TUNNEL_PROCESS
    global PUBLIC_URL


    kill_pattern(
        "cloudflared"
    )

    time.sleep(2)


    for attempt in range(
        1,
        4
    ):

        print(
            "\nCloudflare attempt",
            f"{attempt}/3"
        )


        TUNNEL_LOG.write_text(
            ""
        )


        log_handle = open(
            TUNNEL_LOG,
            "w"
        )


        TUNNEL_PROCESS = subprocess.Popen(
            [
                str(
                    CLOUDFLARED
                ),

                "tunnel",

                "--no-autoupdate",

                "--url",

                (
                    "http://127.0.0.1:"
                    f"{JOB_API_PORT}"
                )
            ],

            stdout=
                log_handle,

            stderr=
                subprocess.STDOUT
        )


        print(
            "Tunnel PID:",
            TUNNEL_PROCESS.pid
        )


        candidate = None


        for _ in range(
            90
        ):

            if (
                TUNNEL_PROCESS.poll()
                is not None
            ):

                break


            time.sleep(1)


            log_text = TUNNEL_LOG.read_text(
                errors="ignore"
            )


            matches = re.findall(
                (
                    r"https://"
                    r"[A-Za-z0-9-]+"
                    r"\.trycloudflare\.com"
                ),
                log_text
            )


            if matches:

                candidate = (
                    matches[-1]
                )

                break


        if not candidate:

            kill_pattern(
                "cloudflared"
            )

            time.sleep(2)

            continue


        print(
            "Candidate:"
        )

        print(
            candidate
        )


        for check in range(
            45
        ):

            if (
                TUNNEL_PROCESS.poll()
                is not None
            ):

                break


            try:

                response = requests.get(
                    candidate +
                    "/health",
                    timeout=10
                )


                print(
                    f"Tunnel check {check + 1}:",
                    response.status_code
                )


                if (
                    response.status_code
                    ==
                    200

                    and

                    response.json().get(
                        "ok"
                    )
                    is True
                ):

                    PUBLIC_URL = candidate


                    print(
                        "✅ HEALTHY PUBLIC TUNNEL"
                    )

                    print(
                        PUBLIC_URL
                    )


                    return PUBLIC_URL


            except Exception:

                print(
                    f"Tunnel check {check + 1}: waiting..."
                )


            time.sleep(3)


        kill_pattern(
            "cloudflared"
        )


        time.sleep(3)


    print(
        tail(
            TUNNEL_LOG,
            220
        )
    )


    raise RuntimeError(
        "Could not establish healthy Cloudflare tunnel."
    )


start_tunnel()


# ======================================================================
# MODEL INFO
# ======================================================================

def get_current_model_info():

    global MODEL_INFO


    try:

        state_response = requests.get(
            LOCAL_BASE +
            "/state",
            timeout=5
        )


        if state_response.ok:

            state_data = (
                state_response.json()
            )


            if (
                state_data.get(
                    "generation_active"
                )

                or

                state_data.get(
                    "model_switching"
                )
            ):

                return MODEL_INFO


        response = requests.get(
            LOCAL_BASE +
            "/model-info",
            timeout=15
        )


        if response.ok:

            info = response.json()


            if (
                info.get(
                    "loaded"
                )
                is True
            ):

                MODEL_INFO = info


    except Exception:

        pass


    return MODEL_INFO


# ======================================================================
# BACKEND CONFIG
# ======================================================================

def make_backend_config():

    info = (
        get_current_model_info()
        or
        {}
    )


    return {

        "backend_url":
            PUBLIC_URL,

        "health_endpoint":
            PUBLIC_URL +
            "/health",

        "model_info":
            PUBLIC_URL +
            "/model-info",

        "models_endpoint":
            PUBLIC_URL +
            "/models",

        "backend_state_endpoint":
            PUBLIC_URL +
            "/state",

        "api_docs":
            PUBLIC_URL +
            "/docs",

        "job_endpoint":
            PUBLIC_URL +
            "/jobs",

        "job_status_template":
            PUBLIC_URL +
            "/jobs/{job_id}",

        "job_audio_template":
            PUBLIC_URL +
            "/jobs/{job_id}/audio",

        "model_switch_job_endpoint":
            PUBLIC_URL +
            "/model-switch-jobs",

        "model_switch_status_template":
            PUBLIC_URL +
            "/model-switch-jobs/{switch_id}",

        "reference_upload_endpoint":
            PUBLIC_URL +
            "/upload_reference",

        "reference_files_endpoint":
            PUBLIC_URL +
            "/get_reference_files",

        "predefined_voices_endpoint":
            PUBLIC_URL +
            "/get_predefined_voices",

        # NEW
        "voice_share_endpoint":
            PUBLIC_URL +
            "/share-voice",

        "tts_mode":
            "async-job",

        "model_switch_mode":
            "async-job",

        "model_loaded":
            info.get(
                "loaded",
                True
            ),

        "model_type":
            info.get(
                "type"
            ),

        "device":
            info.get(
                "device"
            ),

        "sample_rate":
            info.get(
                "sample_rate"
            ),

        "supports_paralinguistic_tags":
            info.get(
                "supports_paralinguistic_tags",
                False
            ),

        "available_paralinguistic_tags":
            info.get(
                "available_paralinguistic_tags",
                []
            ),

        "supports_multilingual":
            info.get(
                "supports_multilingual",
                False
            ),

        "supported_languages":
            info.get(
                "supported_languages",
                {
                    "en":
                        "English"
                }
            ),

        "turbo_available":
            info.get(
                "turbo_available_in_package",
                False
            ),

        "multilingual_available":
            info.get(
                "multilingual_available_in_package",
                False
            ),

        "chatterbox_port":
            CHATTERBOX_PORT,

        "job_api_port":
            JOB_API_PORT,

        "updated_at":
            datetime.now(
                timezone.utc
            ).isoformat()
    }


# ======================================================================
# PUBLISH BACKEND CONFIG
# ======================================================================

def publish_backend_config():

    global LAST_PUBLISHED_MODEL


    github_api = (
        "https://api.github.com/repos/"
        f"{GITHUB_OWNER}/"
        f"{GITHUB_REPO}/contents/"
        f"{GITHUB_FILE}"
    )


    config_data = make_backend_config()


    LAST_PUBLISHED_MODEL = (
        config_data.get(
            "model_type"
        )
    )


    encoded = base64.b64encode(
        json.dumps(
            config_data,
            indent=2
        ).encode(
            "utf-8"
        )
    ).decode(
        "utf-8"
    )


    for attempt in range(
        1,
        4
    ):

        existing = requests.get(
            github_api,

            headers=
                GH_HEADERS,

            params={
                "ref":
                    GITHUB_BRANCH
            },

            timeout=20
        )


        payload = {

            "message":
                (
                    "Auto-update live "
                    "Youtube Blue backend"
                ),

            "content":
                encoded,

            "branch":
                GITHUB_BRANCH
        }


        if existing.status_code == 200:

            payload[
                "sha"
            ] = existing.json()[
                "sha"
            ]


        elif existing.status_code != 404:

            raise RuntimeError(
                "Could not inspect backend-config.json."
            )


        result = requests.put(
            github_api,

            headers=
                GH_HEADERS,

            json=
                payload,

            timeout=30
        )


        print(
            "Github config write:",
            result.status_code
        )


        if result.status_code in (
            200,
            201
        ):

            print(
                "✅ backend-config.json published."
            )

            return


        if result.status_code in (
            409,
            422
        ):

            time.sleep(2)

            continue


        print(
            result.text[:3000]
        )


        raise RuntimeError(
            "Github config update failed."
        )


    raise RuntimeError(
        "Github config update failed after retries."
    )


# ======================================================================
# PUBLISH
# ======================================================================

print(
    "\n[9/10] Publishing backend configuration..."
)


publish_backend_config()


# ======================================================================
# PUBLIC TESTS
# ======================================================================

print(
    "\n[10/10] Testing public API..."
)


PUBLIC_ROUTES = [

    "/health",

    "/model-info",

    "/models",

    "/state",

    "/get_predefined_voices",

    "/get_reference_files"
]


for route in PUBLIC_ROUTES:

    response = requests.get(
        PUBLIC_URL +
        route,
        timeout=30
    )


    print(
        route,
        "→",
        response.status_code
    )


    if response.status_code != 200:

        print(
            response.text[:2500]
        )

        raise RuntimeError(
            "Public route failed: "
            + route
        )


# Public CORS verification for share route.
public_preflight = requests.options(
    PUBLIC_URL +
    "/share-voice",

    headers={
        "Origin":
            "https://zayaiken21.github.io",

        "Access-Control-Request-Method":
            "POST"
    },

    timeout=20
)


print(
    "/share-voice OPTIONS →",
    public_preflight.status_code
)


if public_preflight.status_code not in (
    200,
    204
):

    raise RuntimeError(
        "Public share-voice CORS check failed."
    )


# ======================================================================
# READY
# ======================================================================

banner(
    "✅ YOUTUBE BLUE BACKEND FULLY LIVE"
)


print(
    "\nActive model:",
    MODEL_INFO.get(
        "type"
    )
)


print(
    "Device:",
    MODEL_INFO.get(
        "device"
    )
)


print(
    "\nPublic backend:"
)

print(
    PUBLIC_URL
)


print(
    "\nVoice share endpoint:"
)

print(
    PUBLIC_URL +
    "/share-voice"
)


print(
    "\nAPI docs:"
)

print(
    PUBLIC_URL +
    "/docs"
)


print(
    "\nPublished config:"
)

print(
    PAGES_CONFIG
)


print(
    "\nYoutube Blue:"
)

print(
    FRONTEND_URL
)


print()
print(
    "Leave this cell running."
)

print(
    "Automatic watchdog is active."
)


# ======================================================================
# WATCHDOG
# ======================================================================

public_failures = 0

last_environment_check = 0


while True:

    try:

        time.sleep(30)


        # --------------------------------------------------------------
        # ENVIRONMENT SELF-CHECK
        #
        # Do this every five minutes rather than importing Torch every
        # 30 seconds.
        # --------------------------------------------------------------

        now_ts = time.time()


        if (
            now_ts
            -
            last_environment_check
            >=
            300
        ):

            last_environment_check = now_ts


            healthy, env_details = environment_test()


            if not healthy:

                print(
                    "⚠️ Python environment became unhealthy."
                )


                kill_pattern(
                    "cloudflared"
                )

                kill_pattern(
                    "chatterbox_job_api"
                )

                kill_port(
                    JOB_API_PORT
                )

                kill_pattern(
                    "server.py"
                )

                kill_port(
                    CHATTERBOX_PORT
                )


                repair_environment()


                # Restore Turbo startup configuration.
                with open(
                    CONFIG_PATH,
                    "r"
                ) as f:

                    repair_config = (
                        yaml.safe_load(f)
                        or
                        {}
                    )


                repair_config.setdefault(
                    "model",
                    {}
                )


                repair_config[
                    "model"
                ][
                    "repo_id"
                ] = "chatterbox-turbo"


                repair_config.setdefault(
                    "tts_engine",
                    {}
                )


                repair_config[
                    "tts_engine"
                ][
                    "device"
                ] = "cpu"


                with open(
                    CONFIG_PATH,
                    "w"
                ) as f:

                    yaml.safe_dump(
                        repair_config,
                        f,
                        sort_keys=False
                    )


                start_chatterbox()
                start_wrapper()
                start_tunnel()
                publish_backend_config()


                public_failures = 0

                continue


        # --------------------------------------------------------------
        # CHATTERBOX PROCESS
        # --------------------------------------------------------------

        if (
            SERVER_PROCESS is None

            or

            SERVER_PROCESS.poll()
            is not None
        ):

            print(
                "⚠️ Chatterbox stopped. Restarting..."
            )


            start_chatterbox()
            publish_backend_config()


        # --------------------------------------------------------------
        # WRAPPER PROCESS
        # --------------------------------------------------------------

        if (
            JOB_PROCESS is None

            or

            JOB_PROCESS.poll()
            is not None
        ):

            print(
                "⚠️ Youtube Blue API stopped. Restarting..."
            )


            start_wrapper()


        # --------------------------------------------------------------
        # LOCAL HEALTH
        # --------------------------------------------------------------

        try:

            local_health = requests.get(
                LOCAL_BASE +
                "/health",
                timeout=5
            )


            if local_health.status_code != 200:

                raise RuntimeError(
                    "Local wrapper unhealthy."
                )


        except Exception:

            print(
                "⚠️ Local wrapper health failed."
            )


            start_wrapper()


        # --------------------------------------------------------------
        # MODEL CHANGE DETECTION
        # --------------------------------------------------------------

        try:

            state_response = requests.get(
                LOCAL_BASE +
                "/state",
                timeout=5
            )


            state_data = (
                state_response.json()
            )


            idle = (

                not state_data.get(
                    "generation_active"
                )

                and

                not state_data.get(
                    "model_switching"
                )
            )


            if idle:

                current_response = requests.get(
                    LOCAL_BASE +
                    "/model-info",
                    timeout=10
                )


                if current_response.ok:

                    current_info = (
                        current_response.json()
                    )


                    if current_info.get(
                        "loaded"
                    ) is True:

                        current_type = (
                            current_info.get(
                                "type"
                            )
                        )


                        MODEL_INFO = current_info


                        if (
                            current_type
                            !=
                            LAST_PUBLISHED_MODEL
                        ):

                            print(
                                "✅ Model changed:",
                                current_type
                            )


                            publish_backend_config()


        except Exception:

            pass


        # --------------------------------------------------------------
        # CLOUDFLARE PROCESS
        # --------------------------------------------------------------

        tunnel_dead = (

            TUNNEL_PROCESS is None

            or

            TUNNEL_PROCESS.poll()
            is not None
        )


        if tunnel_dead:

            print(
                "⚠️ Cloudflare stopped. Replacing tunnel..."
            )


            old_url = PUBLIC_URL


            start_tunnel()


            if PUBLIC_URL != old_url:

                publish_backend_config()


            public_failures = 0

            continue


        # --------------------------------------------------------------
        # PUBLIC HEALTH
        # --------------------------------------------------------------

        try:

            public_health = requests.get(
                PUBLIC_URL +
                "/health",
                timeout=10
            )


            if (
                public_health.status_code
                ==
                200

                and

                public_health.json().get(
                    "ok"
                )
                is True
            ):

                public_failures = 0


            else:

                public_failures += 1


        except Exception:

            public_failures += 1


        # --------------------------------------------------------------
        # REPLACE DEAD TUNNEL
        # --------------------------------------------------------------

        if public_failures >= 3:

            print(
                "⚠️ Public tunnel failed 3 checks."
            )


            old_url = PUBLIC_URL


            start_tunnel()


            if PUBLIC_URL != old_url:

                print(
                    "New public URL:"
                )

                print(
                    PUBLIC_URL
                )


                publish_backend_config()


            public_failures = 0


            print(
                "✅ Frontend backend configuration updated."
            )


        else:

            now = (
                datetime.now()
                .strftime(
                    "%H:%M:%S"
                )
            )


            print(
                (
                    f"✓ {now} "
                    f"backend monitor healthy"
                ),
                flush=True
            )


    except KeyboardInterrupt:

        print(
            "\nBackend monitor stopped manually."
        )

        break


    except Exception as exc:

        print(
            "\n⚠️ Watchdog error:"
        )

        print(
            repr(
                exc
            )
        )


        print(
            "Retrying watchdog in 30 seconds..."
        )


YOUTUBE BLUE — SELF-HEALING MULTI-MODEL BACKEND

[1/10] Verifying environment...
✅ Installed environment healthy.

[2/10] Checking Github access...
Github status: 200
✅ Github access confirmed.

[3/10] Preparing Chatterbox configuration...
Startup model: chatterbox-turbo
Device: cpu

[4/10] Clearing stale services...
✅ Old processes cleared.

[5/10] Starting Chatterbox...

Starting Chatterbox...
Chatterbox PID: 5172
Loading model... 0s
Loading model... 20s
{
  "loaded": true,
  "type": "turbo",
  "class_name": "ChatterboxTurboTTS",
  "device": "cpu",
  "sample_rate": 24000,
  "supports_paralinguistic_tags": true,
  "available_paralinguistic_tags": [
    "laugh",
    "chuckle",
    "sigh",
    "gasp",
    "cough",
    "clear throat",
    "sniff",
    "groan",
    "shush"
  ],
  "turbo_available_in_package": true,
  "multilingual_available_in_package": true,
  "supports_multilingual": false,
  "supported_languages": {
    "en": "English"
  }
}
✅ CHATTERBOX READY: turbo

[6/10] Building 